In [0]:
from LDCDataAccessLayerPy import KeyVaultManager, SharePointManager, SqlManager, databricks_init
from datetime import datetime, timedelta
from LDCDataAccessLayerPy import databricks_init, DataLakeManagerGen2
from io import BytesIO
import LDCDataAccessLayerPy
#Initiate the secret to access KeyVault secrets
databricks_init(dbutils, 'GO')
sp_mgr = SharePointManager()
sql_mgr = SqlManager()

import logging
logger = spark._jvm.org.apache.log4j
logging.getLogger("py4j").setLevel(logging.ERROR)

import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import ListedColormap
from sklearn.cluster import KMeans
from openpyxl import load_workbook

from datetime import datetime, timedelta
import re

url = "https://ldcom365.sharepoint.com"



## DICT

In [0]:
country_region_mapping = {
    "ALGERIA": "AFRICA",
    'ALEGERIA':'AFRICA',
    "ARGENTINA": "SOUTH AMERICA",
    "BANGLADESH": "ASIA",
    "BELGIUM": "EU",
    "BRAZIL": "SOUTH AMERICA",
    "CHILE": "SOUTH AMERICA",
    "CHINA": "ASIA",
    "COLOMBIA": "SOUTH AMERICA",
    "COSTA RICA": "CENTRAL AMERICA",
    "CUBA": "CENTRAL AMERICA",
    "CYPRUS": "EU",
    "CYPRUS/GREECE": "EU",
    "DENMARK": "EU",
    "EGYPT": "AFRICA",
    "FRANCE": "EU",
    "GERMANY": "EU",
    "GREECE": "EU",
    "GREECE/ISRAEL": "EU",
    "GREECE/ITALY": "EU",
    "HOLLAND": "EU",
    "HOLLAND/GERMANY": "EU",
    "HONDURAS": "CENTRAL AMERICA",
    "INDONESIA": "SE ASIA",
    "IRAN": "ME ASIA",
    "IRELAND": "EU",
    "ISRAEL": "ME ASIA",
    "ISRAEL/GREECE": "EU",
    "ITALY": "EU",
    "IVORY COAST": "AFRICA",
    "JAPAN": "ASIA",
    "JORDAN": "ME ASIA",
    "KENYA": "AFRICA",
    "KOREA": "ASIA",
    "LEBANON": "ME ASIA",
    "LITHUANIA": "EU",
    "MALAYSIA": "SE ASIA",
    "MAURITIUS": "AFRICA",
    "MEXICO": "CENTRAL AMERICA",
    "MOROCCO": "AFRICA",
    "NETHERLANDS": "EU",
    "NETHERLANDS/GERMANY": "EU",
    "NIGERIA": "AFRICA",
    "PANAMA": "CENTRAL AMERICA",
    "PERU": "SOUTH AMERICA",
    "PHILIPPINES": "SE ASIA",
    "PORTUGAL": "EU",
    "PUERTO RICO": "CENTRAL AMERICA",
    "ROMANIA": "EU",
    "RUSSIA": "ASIA",
    "SAF": "AFRICA",
    "SAUDI ARABIA": "ME ASIA",
    "SENEGAL": "AFRICA",
    "SOUTH AFRICA": "AFRICA",
    "SOUTH KOREA": "ASIA",
    "SPAIN": "EU",
    "SYRIA": "ME ASIA",
    "TAIWAN": "ASIA",
    "TBC": "TBC",
    "THAILAND": "SE ASIA",
    "TUNISIA": "AFRICA",
    "TURKEY": "ME ASIA",
    "U.ARAB EMIRAT": "ME ASIA",
    "UAE": "ME ASIA",
    "UK": "EU",
    "UNITED KINGDOM": "EU",
    "POLAND": "EU",
    "SLOVENIA": "EU",
    "LITUANIA": "EU",
    "CROATIA": "EU",
    "GREECE + CYPRUS":'EU',

    "UNITED ARAB EMIRATES": "ME ASIA",
    "URUGUAY": "SOUTH AMERICA",
    "USA": "NORTH AMERICA",
    "UNITED STATES": "NORTH AMERICA",
    "VENEZUELA": "SOUTH AMERICA",
    "VIETNAM": "SE ASIA",
    "YEMEN": "ME ASIA",
    "": "NOT AVAILABLE",
    "MOZAMBIQUE": "AFRICA",
    "GUATEMALA": "CENTRAL AMERICA",
    "US": "NORTH AMERICA",
    "Z. OTHER EAST AFRICA": "AFRICA",
    "DOMINICAN REPUBLIC": "CENTRAL AMERICA",
    "AUSTRALIA": "OCEANIA",
    "NEW ZEALAND": "OCEANIA",
    "EL SALVADOR": "CENTRAL AMERICA",
    "NICARAGUA": "CENTRAL AMERICA",
    "OMAN": "ME ASIA",
    "KUWAIT": "ME ASIA",
    "SWITZERLAND": "EU",
    "IRAQ": "ME ASIA",
    "HAITI": "CENTRAL AMERICA",
    "CANADA": "NORTH AMERICA",
    "TRINIDAD": "CENTRAL AMERICA",
    "JAMAICA": "CENTRAL AMERICA",
    "ANGOLA": "AFRICA",
    "Z. OTHER FSU": "EUROPE",
    "NORWAY": "EU",
    "NOT AVAILABLE": "",
    "ECUADOR":"SOUTH AMERICA",
    'GUYANA':"SOUTH AMERICA",
    'TANZANIA':'AFRICA',
    'LUANDA':'AFRICA',
    'LIBYA':'AFRICA', 
    'PHILIPPINNES':'SE ASIA',
    'REUNION ISLAND':'AFRICA',
    'GHANA':'AFRICA',
    'CONGO':'AFRICA',
    'NAMIBIA':'AFRICA',
    'CAPE VERDE':'AFRICA',
    'MAURITANA':'AFRICA',
    'UGANDA':'AFRICA',
    'ZIMBABWE':'AFRICA',
    'MALI':'AFRICA',
    'SUDAN':'AFRICA',
    'MAURITANIA':'AFRICA', 
    'ETHIOPIA':'AFRICA',
    'RUANDA':'AFRICA',
    'RWANDA':'AFRICA',
    'BURUNDI':'AFRICA', 
    'LYBIA':'AFRICA',
    'MAURITUS IS':'AFRICA',
    'LATVIA':'EU',
    'ESTONIA':'EU',

    'CAMEROON':'AFRICA',
    'IVORY COST':'AFRICA',
    'REUNION':'AFRICA',
    'DOM. REP':'CENTRAL AMERICA',
    'DOM REP.':'CENTRAL AMERICA',
    'DOM REP;':'CENTRAL AMERICA',
    'DOM. REP.':'CENTRAL AMERICA',
    'DOM. REP;':'CENTRAL AMERICA',
    'DOM REP':'CENTRAL AMERICA',
    'TRINIDAD & TOBAGO':'CENTRAL AMERICA',
    'GEORGIA':'ME ASIA',
    'KUWEIT':'ME ASIA',
    'BAHREIN':'ME ASIA',
    'DJBOUTI':'ME ASIA',
    'SAUDI ARABIA ':'ME ASIA',

    
    'BRUNEI':'SE ASIA',
    'MYANMAR':'SE ASIA',
    'MALAYSIA':'SE ASIA',
    'Malaysia':'SE ASIA',
    'PHILIPINES':'SE ASIA',
    'INDIA':'ASIA',
    'BELARUS':'ASIA',

    'NEW ZELAND':'OCEANIA',
    'MAURITUIS':'AFRICA',
    'U.A.E.':'ME ASIA',
    'EAU':'ME ASIA',
    'LEBANNON':'ME ASIA',
    'HOLANDA':'EU',
    'PAKISTAN':'ASIA',
    'PAKISTAN ':'ASIA',
    'RUSSIAN FEDERATION':'ASIA',
    'UNITED STATES':'ASIA',
    'TURKEY ':'ME ASIA',

    'SOUTH KOREA ':'ASIA',
    'JAPAN  ':'ASIA',
    'MALAYSIA  ':'SE ASIA',
    'IRAK':'ME ASIA', 
    'BRASIL':'SOUTH AMERICA',
    'MARRUECOS':'AFRICA',
    'ECUADOR ':'SOUTH AMERICA',
    'PARAGUAY':'SOUTH AMERICA',
    'BRAZIL ':'SOUTH AMERICA',
    'CHILE ':'SOUTH AMERICA',
    'BOLIVIA':'SOUTH AMERICA',
    'MADAGASCAR':'AFRICA',
    'GABON':'AFRICA',
    'SENEGAL ':'AFRICA',
    'GAMBIA':'AFRICA',
    'QATAR':'ME ASIA', 
    'BAHRAIN':'ME ASIA',
    'LEBANON ' :'ME ASIA',
    'BURKINA FASO':'AFRICA',
    'ALGERIA ':'AFRICA',
    'MALAWI':'AFRICA',
    'GUINEA':'AFRICA',
    'TOGO':'AFRICA',
    'LIBERIA':'AFRICA',
    'DJIBOUTI':'AFRICA',
    'VIETNAM ':'SE ASIA',
    }

origins_mapping={
    'ARGENTINA':'ARG',
    'URUGUAY':'UGY',
    'PARAGUAY':'PYG',
    
    'ARG':'ARG',
    'PY.':'PYG',
    'PY':'PYG',
    'PY ':'PYG',
    'URU':'UGY',
    'UY':'UGY',
    'BOL':'BOL',
    'BZ':'BRA',   }

port_mapping = {
    'SAN LORENZO': 'San Lorenzo',
    'TOYOTA': 'San Lorenzo',
    'San Lorenzo - Argentina':'San Lorenzo',
    'Cofco Int.':'San Lorenzo',
    'PORT':'San Lorenzo',

    
    'ROSARIO': 'Rosario',
    'SAN PEDRO': 'San Pedro',
    'San Pedro - Argentina':'San Pedro',
    
    'NUEVA PALMIRA(R.O.U.)':'Uruguay',
    'NUEVA PALMIRA':'Uruguay',
    'PALMIRA':'Uruguay',
    'Neuva Palmira':'Uruguay',
    'URUGUAY':'Uruguay',

    'BAHIA BLANCA': 'Bahia Blanca',
    'Bahia Blanca - Argentina':'Bahia Blanca',
    'Bahía Blanca - Argentina':'Bahia Blanca',
    'Bahía Blanca':'Bahia Blanca',
    

    'NECOCHEA': 'Necochea',
    'NECOCHEA / QUEQUEN':'Necochea',
    'necochea / quequen':'Necochea',
    'Necochea - Argentina':'Necochea',

    'BUENOS AIRES':'Buenos Aires',
    'SAN NICOLAS': 'San Nicolas',
    'ZARATE':'Zarate',
    'GUAZU': 'Zarate',
    'Guazú': 'Zarate',
    'Zárate':'Zarate',
    'PARANA GUAZU':'Zarate',
    'DEL GUAZU':'Zarate',
    'Paraná Guazú - Argentina':'Zarate',
    'DELTA':'Zarate',
    'LIMA':'Zarate',
    'Lima - Argentina':'Zarate',
    'LAS PALMAS':'Zarate',
    'Zárate - Argentina':'Zarate',
    'Parana Guazu - Argentina':'Zarate',
    'Zarate - Argentina':'Zarate',
    'IBICUY':'Zarate',

    'MONTEVIDEO': 'Uruguay',
    'Montevideo':'Uruguay',
    'MONTEVIDEO (R.O.U)':'Uruguay',
    'MONTEVIDEO(R.O.U)':'Uruguay',
    'FRAY BENTOS':'Uruguay',
    'PAYSANDU':'Uruguay',

    'CAMPANA': 'Campana',
    'Campana - Argentina': 'Campana',
    'campana': 'Campana',

    'GRAL LAGOS':'Rosario',
    'GRAL. LAGOS':'Rosario',
    'LAGOS':'Rosario',
    'Rosario - Argentina':'Rosario',
    'Villa Constitución - Argentina':'Rosario',
    'VILLA CONSTITUCION':'Rosario',
    'VCONSTITUCION':'Rosario',
    'V.CONSTITUCION':'Rosario',
    'Villa Constitucion - Argentina':'Rosario',
    'TIMBUES':'Rosario',
    'ARROYO SECO':'Rosario',
    
    'ESCOBAR':'Escobar',
    
    'SANTA FE':'Santa Fe',
    'CONCEP. DEL URUGUAY':'Concep. Del Uruguay',

    'DIAMANTE':'Diamante',

    'RAMALLO':'Ramallo',
    'RANALLO':'Ramallo',
    'Ramallo - Argentina':'Ramallo',

    'RIO GRANDE':'Rio Grande',
    
    'ALIANZA G2':'Alianza',
    'TRANSHIPMENT':'Transhipment',
    'SAN NICOLÁS':'San Nicolas',
    'San Nicolas - Argentina':'San Nicolas',
    'PORT':'PORT'
}
country_mapping={
    'AUSTRALIA + NEW ZEALAND':'AUSTRALIA',
    'DJBOUTI':'DJIBOUTI',
    'BRASIL':'BRAZIL',
    'BAHREIN':'BAHRAIN',
    'DOM. REP':'DOM REP',
    'DOM REP.':'DOM REP',
    'DOM. REP;':'DOM REP',
    'DOM. REP.':'DOM REP',
    'DOM. REP;':'DOM REP',
    'DOM.REPUBLIC':'DOM REP',
    'DOMINICAN REP':'DOM REP',
    'DOMINICAN REPUBLIC':'DOM REP',
    'CONGO DEM.REP.':'DOM REP',
    'DOM. REPUBLIC':'DOM REP',
    'DOMINICAN REP.':'DOM REP',
    'U.A.E.':'UAE',
    'EAU':'UAE',
    'U. ARAB EMIRATES':'UAE',
    'UNITED ARAB EMIRATES':'UAE',
    'U.ARAB EMIRAT':'UAE',
    'KUWAIT+UNITED ARAB EMIRATES':'UAE',
    'HOLANDA':'NETHERLANDS',
    'HOLLAND':'NETHERLANDS',
    'MAURITUIS':'MAURITIUS',
    'LEBANNON':'LEBANON',
    'US':'USA',
    'LITHUANIA':'LITUANIA',
    'Malaysia':'MALAYSIA',
    'IVORY COST':'IVORY COAST',
    'MALASYA':'MALAYSIA',
    'UNITED KINGDOM':'UK',
    'U.KINGDOM':'UK',
    'MAURITIUS ISL.':'MAURITIUS',
    'MAURITIUS IS':'MAURITIUS',
    'CHILE+PERU':'CHILE',
    'PTO RICO':'PUERTO RICO',
    'PTO. RICO':'PUERTO RICO',
    'COSTA RICA+GUATEMALA':'COSTA RICA',
    'CHLE':'CHILE',
    'EUROPEAN PORTS':'EU',
    'ALEGERIA':'ALGERIA',
    'NUEVA PALMIRA':'URUGUAY',
    'PTO RICO/DOM REP':'DOM REP',
    'KUWUAIT':'KUWAIT',
    'KWUEIT':'KUWAIT',
    'OMAN+KUWAIT+SAUDI ARABIA':'KUWAIT',
    'PERÚ':'PERU',
    'PERU/ECUADOR':'PERU',
    'DEMOCRATIC REPUBLIC OF THE CONGO':'CONGO',
    'REPUBLIC OF THE CONGO':'CONGO',
    'MATADI':'CONGO',
    'POINTE NOIRE':'CONGO',
    'DAKAR':'SENEGAL',
    'TAILANDIA':'THAILAND',
    'GREECE/ISRAEL':'GREECE',
    'GREECE/ITALY':'GREECE',
    'HOLLAND/GERMANY':'GERMANY',
    'IRAQ':'IRAK',
    'ISRAEL/GREECE':'ISRAEL',
    'MAURITANA':'MAURITANIA',
    'TRINIDAD & TOBAGO':'TRINIDAD',
    'PHILIPPINNES':'PHILIPPINES',
    'PHILIPINES':'PHILIPPINES',
    'MAURITUS IS':'MAURITIUS',
    'CYPRUS/GREECE':'GREECE',
    'NEW ZELAND':'NEW ZEALAND',
    'RUSSIAN FEDERATION':'RUSSIA',
    'SAUDI ARABIA + UNITED ARAB EMIRATES + OMAN':'SAUDI ARABIA'
    
    }   

month_mapping = {
    1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun',
    7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'
}

shippers_mapping={

 'ACSA': 'DACSA',
 'ADECO AGROP.': 'ADECO AGROP',
 'ADECOAGRO': 'ADECO AGROP',
 'ADECO': 'ADECO AGROP',
 'AGRICORP': 'AGROCORP',
 'ALEA & CIA':'ALEA',
 'ADM ARG': 'ADM',
 'ADM AGRO': 'ADM',
 'ADM PAR': 'ADM',
 'ADM PY': 'ADM',
 'ADM, TBC': 'ADM',


 'AGP-GRAIN': 'AGP',
 'AGP GRAIN':'AGP',
 'AGRIBUSINES': 'AGRIBUSINESS',
 'AGRO BUSINESS': 'AGRIBUSINESS',
 'AGRO GAUCHO SA': 'AGRO GAUCHO',
 'AGRO INDUSTRIAS BAIRES': 'AGROINDUSTRIAS BAIRES',
 'AGROFINA S.A.': 'AGROFINA SA',
 'AGROINVERSIONES PAMPEANAS': 'AGRO INVERSIONES PAMPEANAS',
 'AGRONEGOCIOS JEWELL': 'AGRONEGOCIOS JEWEL SRL',
  'AGROEAN': 'AGROCEAN',
 'AGROOCEAN': 'AGROCEAN',
 'AGROPOR': 'AGROPORT',
 'ALG.AVELLANEDA': 'ALG. AVELLANEDA',
 'ALGHURAIR': 'AL GHURAIR',
 'ALIMENTOS & FORRA': 'ALIMENTOS',
 'ALIMIENTOS': 'ALIMENTOS',

 'ALMARAIS': 'ALMARAI',
 'ALNARAI': 'ALMARAI',
 'AMAGGI ': 'AMAGGI',
 'AMGGI': 'AMAGGI',
 'AMDERSONS': 'ANDERSONS',
 'APL CODRICO': 'APL-CODRICO',
 'BGH ': 'BGH',
 'BIO GRAIN': 'BIOGRAIN',
 'BIO-OILS': 'BIO OILS',
 'BOHAI': 'BOHI',
 'BRAINCORP': 'GRAINCORP',
 ' BUNGE': 'BUNGE',
 'BUINGE': 'BUNGE',
 'BUNE': 'BUNGE',
 'BUNGE BOLIV': 'BUNGE',
 'BUNGE GVA': 'BUNGE',
 'BUNGE PAR': 'BUNGE',
 'BUNGE PARAGUAY': 'BUNGE',
 'BUNGE PARAG': 'BUNGE',

 'C.CEREALES': 'CAM CEREALES',
 'CAN CEREALES': 'CAM CEREALES',
 'CAM ': 'CAM CEREALES',
 'CAFI': 'CAF',
 'CALVESTONE': 'CALVESTON',
 'CAMPOS VERDES ARGENTINOS': 'CAMPOS VERDES ARGENTINOS S.A.',
 
 'CARGIILL': 'CARGILL',
 'CARGIL': 'CARGILL',
 'CARGIL BOL': 'CARGILL',
 'CARGILL  ': 'CARGILL',
 'CARGILL UY': 'CARGILL',
 'CARGLL': 'CARGILL',
 'CARILL': 'CARGILL',
 'CARGILL AGROP':'CARGILL',
 'CARGILL AMERICAS INC.':'CARGILL',
 'CARGILL BOLIV':'CARGILL',
 'CARGILL INCORP.':'CARGILL',
 'CARGILL AGROP':'CARGILL',

 'CASILO': 'CASILLO',
 'CASSILO': 'CASILLO',
 'CASTILLO': 'CASILLO',

 'CC AGRO & TEC S.A.': 'C C AGRO & TEC S.A',
 'CEREAL DOCKS': 'CEREAL DOCK',
 'CEREALERA AZUL': 'CERALERA AZUL',
 'CEREALES MAGGIOLLO': 'CEREALES MAGGIOLO',
 'CEREALIS ': 'CEREALIS',
 'CEREOIL ': 'CEREOIL',
 'CERFLOY': 'CERFOLY',
 'CHINA AGRI OIL': 'CHINA AGRI OILS',
 'CHINA AGRIOILS': 'CHINA AGRI OILS',
 'CHS DE ARGENTINA': 'CHS',
 'CHS ARGENTINA': 'CHS',
 'CHS DE URUGUAY': 'CHS',

 'CÍA ARG DE GRANOS': 'CIA ARG DE GRANOS',
 'CLP CHEMICALS ': 'CLP CHEMICALS',

 'COCFCO': 'COFCO',
 'COFCO RESOURCES SA': 'COFCO',
 'COFCO AGRI ARG':'COFCO',
 'COFCO INT.':'COFCO',
 'COFCO INT. ARG.':'COFCO',
 'COFCO. ERRO':'COFCO',
 'COFCO INT.ARG':'COFCO',

 'COGENTRA': 'CONGENTRA',
 'CONTEGRAL ': 'CONTEGRAL',
 'COOP AVEL': 'COOP AVE',
 'COPAGRAN': 'COPAGRA',
 'CORP CEREALES': 'CORP. DE CEREALES',
 'CORP. CEREALES': 'CORP. DE CEREALES',
 'CORVEN MOTORS': 'CORVEN MOTOR',
 'CROSLAND': 'CROSSLAND',
 'CURICJA': 'CURCIJA',
 'CURSIJA': 'CURCIJA',
 'DAEWOO': 'DAEWO',
 'DERIVADOS VÍNICOS SA': 'DERIVADOS VINICOS',
 'DESA': 'DESAB',
 'DIAZ & FORTI SRL': 'DIAZ & FORTI',
 'DIAZ FORTI': 'DIAZ & FORTI',
 'DIAZ Y FORTI': 'DIAZ & FORTI',
 'DIMITRIAKI ': 'DIMITRIAKI',
 'E-GRAIN': 'E GRAIN',
 'EGRAIN': 'E GRAIN',
 'E.CRUSHING': 'ER CRUSHING',
 'E.R. CRUSHING': 'ER CRUSHING',
 'ERCRUSHING': 'ER CRUSHING',
 'ED & FMAN': 'ED & F MAN',
 'ENGLEHART': 'ENGELHART',
 'ERCA': 'ERCSA',
 'EST.SAN PATRICIO': 'SAN PATRICIO',
 'ESTAB.AGROP.DEL SUDESTE SA': 'EST.AGROP DEL SUDESTE',
 'ETA-DUBAI': 'ETA DUBAI',
 'FACCIUTO FERNANEZ': 'FACCIUTO FERNANDEZ',
 'FERERAL AGROPECUARIA': 'FEDERAL AGROPECUARIA',
 'FONDEMONTE S.A.': 'FONDOMONTE',
 'FONDO MONTE': 'FONDOMONTE',
 'FONDOMONTE': 'FODOMONTE',
 'FONDOMONTE SOUTH AMERICA SA': 'FONDOMONTE',
  'FONDOMONTE SOUTH AMERICA': 'FONDOMONTE',

 'GANADERIA INTEGRAL NICARAGUA S.A.': 'GANADERIA INTEGRAL NICARAGUA',
 'GAVILLON': 'GAVILON',
 'GLCENCORE': 'GLENCORE',
 'GLENCONRE': 'GLENCORE',
 'GLENCOR': 'GLENCORE',
 'GM COMMODITIES': 'COMMODITIES',
 'GRAIN CORP': 'GRAINCORP',
 'GRANELES ': 'GRANELES',
 'GRANELES DE CHILE': 'GRANELES',
 'GRANES': 'GRANELES',
 'GRANELES ANDINOS': 'GRANELES',
 'GREEN ENERGY': 'GREENERGY',
 'GROBOCOPATEL HERMANOS': 'GROBOCOPATEL  HNOS',
 'GRVETAL': 'GRAVETAL',
 'IMP.Y EXP DEL NORTE': 'IMP.& EXP DEL NORTE',
 'IN VIVO': 'INVIVO',
 'IND DE ACEITE': 'IND ACEITE',
 'IND. DE ACEITE SA': 'IND DE ACEITE',
 'INDAGRO': 'INAGRO',
 'INGREDION SA': 'INGREDION ARG',
 'INTEPEC': 'INTERPEC',
 'INTERGRAIN': 'INTERGRAIN SA',
 'IOL SA': 'IOL',
 'IOLSA': 'IOL',
 
 'ITACOL': 'ITALCOL',
 'JC INTERNATIONAL': 'CJ INTERNATIONAL',
 'JTB GLOBAL SERVICES': 'JIT GLOBAL SERVICES',
 'LA TRANQUERA VERDE': 'LA TRANQUERA VERDE SA',
 'LDC ': 'LDC',
 'LDC PAR': 'LDC',
 'LDC PY': 'LDC',
 'LDC PARAGUAY': 'LDC',
 'DREYFUS':'LDC',
 'DREYFUS PAR':'LDC',
 'DREYFUS PARAGUAY':'LDC',
 
 'LEVESTOCK': 'LIVESTOCK',
 'LIFESTOCK': 'LIVESTOCK',
 'LIVESTOCK FD': 'LIVESTOCK',
 'LIVESTOCK FEED': 'LIVESTOCK',
 'LOSUR OVERSEAS': 'LOSUR OVERSEAS SL',
 'LUBRIFICANTI': 'LUBIRIFICANTI',
 'LUIS DUCRET': 'LUIS A DUCRET',
 'MALT. PAMPA': 'MALT PAMPA',
 'MALT PAMPA ': 'MALT PAMPA',
 'MANAGRO': 'MAN AGRO',
 'MARINE OLIE': 'MARINE OIL',
 'MALT.QUILMES':'MALT. QUILMES',
 'MED SOFTS': 'MEDSOFTS',
 'MEDSOFT': 'MEDSOFTS',
 'MEERA': 'MERA',
 'MERA INT': 'MERA',

 'MIDLE GRAIN': 'MIDDLE GRAIN',
 'MILCORP': 'MILICORP',
 'MILICORP TRADING': 'MILICORP',
 'MILLCORP': 'MILCORP',
 'MILLCORP TRADING': 'MILICORP',
 'MOHINO PAULISTA': 'MOLINO PAULISTA',
 'MOIHNO PAULISTA': 'MOLINO PAULISTA',
 'MOLINO PAULISTA': 'MOLINO PAULISTA',
 'MOLINO CAÑUEALAS': 'MOLINO CANUELAS',
 'MOLINO CAÑUELAS': 'MOLINO CANUELAS',
 'MOLINOS CAÑUELAS': 'MOLINO CANUELAS',
 'MOLINO PACIFICO': 'MOLINOS PACIFICO',
 'MOLINOS ': 'MOLINOS',
 'MOLINOS AGRO': 'MOLINOS AGRO S.A',
 'MOLINOS AMERICANOS': 'MOLINO AMERICANO',
 'MORENO': 'O.MORENO',

 'NIDERA ': 'NIDERA',
 'NIDERA UY': 'NIDERA',
 'NIIDERA': 'NIDERA',
 'NOBE': 'NOBLE',
 'NOBEGRAIN': 'NOBLE',
 'NOT AVAILABLE': '(NOT AVAILABLE)',
 'NUTIOIL': 'NUTRIOIL',
 'NUTRI OIL': 'NUTRIOIL',
 'NUTRIIOL': 'NUTRIOIL',
 'NUTRIOL': 'NUTRIOIL',
 'NUTROIL': 'NUTRIOIL',

 'O MORENO': 'MORENO',
 'P CREMER': 'CREMER',
 'P. CREMER': 'P CREMER',
 'PAN AMERICAN': 'PANAMERICA',
 'PAN AMERICAN GRAINS': 'PANAMERICA',
 'PANAMERICAN': 'PANAMERICA',
 'PANAMERICANA': 'PANAMERICA',
 'PARAMERICA': 'PANAMERICA',

 'PERDUE AGRIBISINESS': 'PERDUE AGRIBUSINESS',
 'PERDUE AGRIBUSINESS': 'PERDUE AGRIBUSINESS',
 'PETROAGRO SA': 'PETROAGRO',
 'PRO AGRO SA': 'PETROAGRO',
 'PROAGRO': 'PETROAGRO',

 'PHEATON': 'PHAETON',
 'PRADERA NATURAL': 'PRADERA NATURAL SA',
 'PRIMINDS': 'PREMINDS',

 'PTO ARROYO SECO': 'PTO ARROYO SECO SRL',
 'RAUL H PEREZ': 'RAUL H PEREZ',
 'RAUL H.PEREZ': 'RAUL H PEREZ',
 'RECOUP ': 'RECOUP',
 'SALTO AGUARA': 'SALTO AGUARAY',
 'SAN FERNADO': 'SAN FERNANDO',
 'SAN FERNANDA': 'SAN FERNANDO',
 'SCORCIELLO': 'SCORZIELLO',
 'SEABOARD ': 'SEABOARD',
 'SEABORAD': 'SEABOARD',
 'SEABORD': 'SEABOARD',
 'SIRENTZ': 'SIERENTZ',
 'SODRUDESTVO': 'SODRUGESTVO',
 'SODRUGESTIVO': 'SODRUGESTVO',
 'SODRUGETSVO': 'SODRUGESTVO',
 'SODRUGETZVO': 'SODRUGESTVO',
 'SODRUJESTVO': 'SODRUGESTVO',

 'SORPODI': 'SOPRODI',
 'SOYA MIILLS': 'SOYA MILLS',
 'SOYA MILLS ': 'SOYA MILLS',
 'SOYAMILLS': 'SOYA MILLS',
 'SPECIAL GRAIN': 'SPECIAL GRAINS',
 'SPECIAL GRAINS': 'SPECIAL GRAINS',
 'SPECIAL G.': 'SPECIAL GRAINS',
 'SPECIAL GRAINS SA': 'SPECIAL GRAINS',
 
 'TIRIYAKI': 'TIRYAKI',
 'TIRYAKY': 'TIRYAKI',
 'TIRYALI': 'TIRYAKI',
 'TRIYAKI': 'TIRYAKI',
 'TOEPFR': 'TOEPFER',
 'TOPEFER': 'TOEPFER',
 'TORNULAR': 'TORUNLAR',
 'TOYOTA  TSUSHO': 'TOYOTA TSUSHO',
 'UNION AGRIC AVELLANEDA': 'UNION AGRÍCOLA AVELLANEDA ',
 'UNION AGRÍCOLA AVELLANEDA ':'UNION AGRICOLA AVELLANEDA',

 'URCCOPA': 'URCOOPA',
 'URCOOPA PROVAL': 'URCOOPA',
 'URCOPA': 'URCOOPA',
 'VARIOS': 'VARIOUS',
 'VICENTÍN': 'VICENTIN',
 'VILUCO ': 'VILUCO',
 'VITERRA ACOPIO S.A.':'VITERRA',
 'VITERRA ACOPIO S.A':'VITERRA',
 'VITERRA ARGENTINA S.A':'VITERRA',
 'VITERRA ARGENTINA SA':'VITERRA',
 'VITERRA ARGENTINA SA+YPF':'VITERRA',

 'WELLINGTON': 'WILLMINGTON',
 'WILGMINTON': 'WILLMINGTON',
 'YELLOW ROCK': 'YELLOWROCK',
 'YELOWROCK': 'YELLOWROCKK',
 'ZEN-NOH': 'ZEN NOH',
 'ZENNOH': 'ZEN NOH'}

bahia_berth_mapping = {
    'tbb 9':'TBB', # Port is always Bahia (1 san lorenzo)
    'bb 9':'TBB',
    'tbb':'TBB',# Port is always Bahia (1 san lorenzo)
    'tbb9':'TBB',# Port is always Bahia (1 san lorenzo)
    'terminal bahia blanca berths 9':"TBB", 
    'tbb 5/6':'TBB',
    'bb 26':'TBB',
    '5/6':'TBB',
    'adm agro bahía blanca (ex toepfer)':'TBB',
    'adm agro bahía blanca':'TBB',
    'adm agro bahia blanca':'TBB',
    '5 pg':'TBB',
    'pier 5/6':'TBB',
    
    'dreyfus terminal':'LDC Bahia Blanca', #Every port is Bahia
    'dreyfus grain terminal':'LDC Bahia Blanca',
    'ldc bahia blanca':'LDC Bahia Blanca',
    'LDC Bahia Blanca':'LDC Bahia Blanca',
    
    'terminal bahia blanca':'TBB',

    'cargill terminal':'Puerto Ing White', #Port is Bahia
    'cargill (bb)':'Puerto Ing White', 
    'puerto ing white':'Puerto Ing White',
    'cargill':'Puerto Ing White',
    'Puerto Ing White':'Puerto Ing White',
    
    'galvan 2/3':'Puerto Galvan',
    'galvan':'Puerto Galvan',
    'glencore':'Puerto Galvan', #bahia
    'gt/ute':'Puerto Galvan',
    'ute':'Puerto Galvan', #bahia
    'pier 9':'Puerto Galvan', #Bahia
    'omhsa terminal': 'Puerto Galvan', #Bahia
    'viterra puerto galvan':'Puerto Galvan',
    'berths 2 / 3 viterra (ex moreno) puerto galván':'Puerto Galvan',
    '2/3 pg': 'Puerto Galvan',
    'pg 2/3':'Puerto Galvan',
    'puerto galvan':'Puerto Galvan',
    'ohmsa':'Puerto Galvan',

    'adm agro':'ADM Bahia Blanca',
    'el transito':'ADM Bahia Blanca',
    'adm agro (ex toepfer)':'ADM Bahia Blanca',
    'adm agro (ex l.piedra/toepfer)':'ADM Bahia Blanca',
    'adm':'ADM Bahia Blanca',
    'toepfer':'ADM Bahia Blanca',
    'l.piedrabuena/toepfer termin':'ADM Bahia Blanca'
}
 
rosario_berth_mapping={
    'punta alvear': 'Punta Alvear',
    'punta': 'Punta Alvear',
    'p.alvear':'Punta Alvear',
    'cargill punta alvear':'Punta Alvear',
    'Punta Alvear':'Punta Alvear',
    
    'serviport 1':"TPR", #Port is San Pedro --> Elevator?
    'serviport 6':"TPR", #Port is Rosario
    'serviport 7':"TPR", #Port is Rosario
    'serviport 2':'TPR', #Port is Villa constitucion
    'serviport 3':'TPR', # 1 row, rosario
    'servicios portuarios':'TPR', #Villa constitucion
    'serv.port.':'TPR',#Villa constitucion
    'serv port':'TPR',#Villa constitucion
    'servicios portuarios (unit 2)':'TPR',
    'villa constitucion':'TPR',
    'terminal puerto rosario (open berth) new mole south':'TPR',
    'u6':'TPR', #Rosario
    'u7':'TPR', #Rosario
    'unit 6':'TPR',
    'unidad 6':'TPR', # Rosario
    'unidad 7':'TPR',# Rosario
    'unit vii':'TPR', #port is Rosario
    'unit vi':'TPR', #port is Rosario
    'unit vi rosario':'TPR',
    'unit vii rosario':'TPR',
    'unidad 9':'TPR', 
    'unidad 11':'TPR', 
    'unidad 10':'TPR',
    'unidad 12':'TPR',
    'unidad 8':'TPR',
    'u.6':'TPR',
    'unit 6':'TPR',
    'unit 7':'TPR',
    'terminal puerto':'TPR',
    'm.nvo-s': 'TPR',
    'm.nvo-n': 'TPR',
    "new mole south (terminal puerto rosario)": "TPR",
    "NEW MOLE SOUTH (TERMINAL PUERTO ROSARIO)": "TPR",
    "tpr":"TPR",
    'new port':'TPR',
    'TPR':'TPR',
    'arroyo seco':'Arroyo Seco',
    'arroyo':'Arroyo Seco',
    'term.a.seco':'Arroyo Seco',
    'adm agro - arroyo seco':'Arroyo Seco',
    'term. a. seco':'Arroyo Seco',
    'arroyo seco terminal':'Arroyo Seco',
    'toepfer arroyo seco':'Arroyo Seco',
    'adm agro arroyo seco (ex a.seco terminal)':'Arroyo Seco',
    'adm agro arroyo seco':'Arroyo Seco',
    'adm agro arroyo seco (ex toepfer)':'Arroyo Seco',
    'adm agro (ex toepfer)':'Arroyo Seco',
    'toepfer':'Arroyo Seco',
    'adm':'Arroyo Seco',
    'adm agro san lorenzo':'Arroyo Seco',

    
    'lagos':'LDC Gral Lagos',
    'dreyfus terminal gral. lagos':'LDC Gral Lagos',
    'gl':'LDC Gral Lagos',
    'dreyfus dry cargo terminal':'LDC Gral Lagos', 
    'ldc gral lagos': 'LDC Gral Lagos',
    'dreyfus vegoil terminal': 'LDC Gral Lagos',
    'gral.lagos': 'LDC Gral Lagos',
    'LDC Gral Lagos': 'LDC Gral Lagos',
    'dreyfus vgoil berth gral. lagos':'LDC Gral Lagos',
    'dreyfus liquids terminal':'LDC Gral Lagos',

    'vgg':'VGG',
    'galvez':'VGG',
    'vgg term': 'VGG',
    "villa gobernador galvez": "VGG",
    'v.g.g':'VGG',
    'v.g.g.':'VGG',
    'crane':'VGG',
    'floating crane':'VGG',
    'cargill':'VGG',
    'VGG':'VGG',
    'cargill vgg':'VGG'
}

san_lorenzo_berth_mapping={
    "noble": "Cofco Timbues",
    'timbues noble':"Cofco Timbues",
    'cofco':'Cofco Timbues',
    "noble timbues": "Cofco Timbues",
    "noble floating berth": "Cofco Timbues",
    "cofco agri floating berth": "Cofco Timbues",
    'cofco agri timbues': 'Cofco Timbues',
    'cofco int. timbues':'Cofco Timbues',
    'cofco timbues':'Cofco Timbues',
    'noble timb':'Cofco Timbues',
    'cofco agri':'Cofco Timbues',
    'cofco agri lima (site a)':'Cofco Timbues',
    'cofco int. timbúes':'Cofco Timbues',
    'cofco int. lima (site a)':'Cofco Timbues',
    'Cofco timbues':'Cofco Timbues',
    'Cofco PGSM':'Cofco Timbues',
    'Cofco Timbues':'Cofco Timbues',
    'cofco int. timbues fl.berth':'Cofco Timbues',

    'transito': 'El Transito',
    'el transito (adm agro)':'El Transito',
    'adm': 'El Transito',
    "transito (adm agro)": "El Transito",
    'adm agro san lorenzo (ex tránsito)': "El Transito",
    'adm agro':'El Transito',
    'el transito':'El Transito',
    'adm agro (ex toepfer)':'El Transito',
    'adm agro (ex l.piedra/toepfer)':'El Transito',
    'l.piedrabuena/toepfer termin':'El Transito',
    'transito (adm agro)':'El Transito',
    'adm agro arroyo seco (ex toepfer)':'El Transito',
    'adm agro san lorenzo':'El Transito', 
    'molca':'El Transito',
    'tránsito':'El Transito',
    'toepfer':'El Transito',

    "quebracho": "Quebracho",
    "qubracho": "Quebracho",
    

    'dempa': 'Bunge PGSM',
    'pampa': 'Bunge PGSM',
    'pampra': 'Bunge PGSM',
    
    'bunge grain':'Bunge PGSM',
    "bunge terminal": "Bunge PGSM",
    "bunge": "Bunge PGSM",
    'buenge terminal':'Bunge PGSM',
    "bunge fert": "Bunge PGSM",
    'bunge pgsm':'Bunge PGSM',

    "nidera": "Cofco PGSM",
    'cofco pgsm': 'Cofco PGSM',
    'cofco intl. south berth ex noble)':'Cofco PGSM',
    'cofco intl pgsm south berth          (ex nidera )':'Cofco PGSM',
    'cofco intl. south berth (ex noble)':'Cofco PGSM',
    'cofco argentina south berth':'Cofco PGSM',
    'cofco intl. south berth':'Cofco PGSM',
    'cofco intl. pgsm south berth (ex nidera)':'Cofco PGSM', 
    'cofco int. pgsm south':'Cofco PGSM',
    "nidera fertilizers": "Cofco PGSM",
    'nidera fert.': 'Cofco PGSM',
    'nidrea':'Cofco PGSM',
    'cofco pgsm south (ex nidera)':'Cofco PGSM',
    'cofco pgsm south':'Cofco PGSM',
    'cofco intl. north berth':'Cofco PGSM',
    'cofco intl. pgsm north berth (ex nidera fertilizantes)':'Cofco PGSM',
    'cofco int. pgsm north':'Cofco PGSM',


    'terminal 6 norte':'T6 - Resinfor',
    'terminal 6 sur':'T6 - Resinfor',
    'terminal 6':'T6 - Resinfor',
    't6':'T6 - Resinfor',
    'arauco argentina':"T6 - Resinfor",
    "term.6-n": "T6 - Resinfor",
    "term.6-s": "T6 - Resinfor",
    'berth 6': 'T6 - Resinfor',
    'resinfor': 'T6 - Resinfor',
    'terminal 6-north':'T6 - Resinfor',
    't6 - resinfor':'T6-Resinfor',
    'terminal 6  sur':'T6-Resinfor',
    'terminal 6 (n) north (s) south berth - tbc':'T6-Resinfor',
    'terminal 6-pier tbc':'T6-Resinfor',
    'terminal 6 (s) south berth':'T6-Resinfor',
    'terminal 6 (n) north berth':'T6-Resinfor',
    'terminal 6-north berth':'T6-Resinfor',
    'terminal 6-south berth':'T6-Resinfor',
    'terminal 6 south berth':'T6-Resinfor',
    'terminal vi':'T6-Resinfor',
    'terminal 6-south':'T6-Resinfor',
    'arauco':'T6-Resinfor',
    'terminal 6 north berth':'T6-Resinfor',

    'a.c.a.':"ACA San Lorenzo",
    'aca':"ACA San Lorenzo",
    'a.c.a':"ACA San Lorenzo",
    'a.c.a. timbues':'ACA San Lorenzo',
    'aca san lorenzo':'ACA San Lorenzo',
    'aca timbues':'ACA San Lorenzo',
    'ACA San Lorenzo':'ACA San Lorenzo',
    'vicentin': 'Vicentin Port',
    'vincentin': 'Vicentin Port',
    'vicentin port': 'Vicentin Port',

    'san benito': 'San Benito',
    'san benigto':'San Benito',
    'sam benito': 'San Benito',
    'vietnam':'San Benito',
    '<':'San Benito',

    'ldc timbúes': 'LDC Timbues',
    "dreyfus timbues": "LDC Timbues",
    'timbues ldc':"LDC Timbues",
    'ldc timbues': 'LDC Timbues',
    'dreyfus vegoil terminal':'LDC Timbues',
    'ldc':'LDC Timbues',
    'lcd':'LDC Timbues',
    'LDC Timbues':'LDC Timbues',
    'LDC timbues':'LDC Timbues',
    'dreyfus terminal':'LDC Timbues',
    'ldc timbúes': 'LDC Timbues',

    'renova (north berth)':'Renova',
    'renova north berth':'Renova',
    'renova south':'Renova',
    'renova north':'Renova',
    'renova term': 'Renova',
    "renova south berth": "Renova",
    'renova':'Renova',
    'renova-north':'Renova',
    'renova-south':'Renova',
    'a.g.d. timbues (aceitera general deheza)':'AGD Timbues',#1 row San lorenzo
    'a.g.d. timbues':'AGD Timbues',
    'agd timbues':'AGD Timbues',
    'AGD Timbues':'AGD Timbues',
    'terminal g':'AGD Timbues',
    'akzo nobel': 'Nouryon - Akzo Nobel',
    'nouryon /ex akzo nobel': 'Nouryon - Akzo Nobel',
    'nouryon - akzo nobel':'Nouryon - Akzo Nobel',
    'nouryon chemicals argentina':'Nouryon - Akzo Nobel',
    
    'minera alumbrera':'T6 - Minera Alumbrera',
    't6 - minera alumbrera':'T6 - Minera Alumbrera'
}

uruguay_berth_mapping={
    'montevideo': 'Montevideo',
    'pier c': 'Montevideo',
    'pier b': 'Montevideo',
    'transgranel':'Montevideo', #Montevideo
    'obrinel':'Montevideo',#Montevideo
    'terminal granelera obrinel':'Montevideo',#Montevideo
    'tgo':'Montevideo', #Montevideo
    'tgm terminal':'Montevideo',
    'tgm':'Montevideo',
    'pier 6-7':'Montevideo',
    'navios':'Nueva Palmira',
    'navios terminal': 'Nueva Palmira', 
    'navios terminal south': 'Nueva Palmira', 
    'navios terminal north': 'Nueva Palmira',
    'terminal navios north': 'Nueva Palmira', 
    'navios pier 1': 'Nueva Palmira',
    'navios pier 3':'Nueva Palmira',
    'navios pier 4':'Nueva Palmira',
    'pier 1-2':'Nueva Palmira',
    'terminal navios south':'Nueva Palmira',
    't.g.u.':'Nueva Palmira', #port always NP
    'tgu':'Nueva Palmira',
    'anp north pier':'Nueva Palmira',
    'tgu/anp north section':'Nueva Palmira',
    'tgu terminal/anp north':'Nueva Palmira',
    'tgu (anp nort section)':'Nueva Palmira',
    'tgu/anp/north section':'Nueva Palmira',
    'anp':'Nueva Palmira',
    'a.g.d. timbues':'Nueva Palmira',
    'tgu pier north':'Nueva Palmira',
    'tgu (anp north section)':'Nueva Palmira',
    'tgu/anp north':'Nueva Palmira',
    'ontur':'Nueva Palmira', #Port is nueva palmira
    'ontur terminal':'Nueva Palmira',
    'antwerpen':'Nueva Palmira', #nueva palmira
    'anw':'Nueva Palmira',#nueva palmira
    'anp/north section':'Nueva Palmira',#nueva palmira
    'fiorucci':'Nueva Palmira',#nueva palmira
    'fiorucci international':'Nueva Palmira',
    'sitio 0':'Nueva Palmira',#nueva palmira
    'transf':'Nueva Palmira', #nueva palmira
    'tgu north pier':'Nueva Palmira',
    'nueva palmira':'Nueva Palmira',
    'paysandu':'Paysandu',
    'fray bentos':'Fray Bentos',
}

## FUNCTIONS

In [0]:
def extract_date_or_default_sbm(row, default_days=0):
    today = datetime.today()
    status_text = row['Status']
    flag = None

    # Check for 'Alongside'
    if 'Alongside' in status_text:
        if today.day >= 28:
            flag = 'Alongside'
        return today + timedelta(days=2), flag

    # Check for 'At roads'
    if 'At roads' in status_text:
        if today.day >= 26:
            flag = 'At roads'
        return today + timedelta(days=4), flag

    if 'At San Nicolas ' in status_text:
        if today.day >= 26:
            flag = 'Alongside'
    
    if 'At Rosario ' in status_text:
        if today.day >= 26:
            flag = 'Alongside'
            
        return today + timedelta(days=4), flag
    # Extract date in dd/mm/yyyy format
    match = re.search(r'\d{2}/\d{2}/\d{4}', status_text)
    if match:
        try:
            return pd.to_datetime(match.group(), format='%d/%m/%Y'), flag
        except ValueError:
            return pd.NaT, flag
    else:
        return today + timedelta(days=default_days), flag


In [0]:
def extract_date_or_default(row, default_days=0):
  """
  Extracts a date from a string or returns a default date.

  Args:
      row: The string to extract the date from.
      default_days: Number of days to add to today's date if no date is found (default: 0).

  Returns:
      A datetime object or NaT if no valid date is found.
  """

  # Check if the row contains "Alongside"
  if 'Alongside' in row:
    return datetime.today() + timedelta(days=5)  # Return today's date if "Alongside" is found

  # Regular expression to find dates in dd/mm/yyyy format (strict year range)
  match = re.search(r'\d{2}/\d{2}/\d{4}', row)
  if match:
    try:
      return pd.to_datetime(match.group(), format='%d/%m/%Y')
    except ValueError:
      # Handle potential parsing errors
      return pd.NaT
  else:
    return datetime.today() + timedelta(days=default_days)
    
    
def extract_date(row):
    for col in ['ETA', 'ETB', 'ETF']:
        if pd.notna(row[col]):
            date_str = row[col][-5:]
            try:
                return pd.to_datetime(date_str + '/2025', format='%d/%m/%Y')
            except ValueError:
                continue
    return np.nan

def rebuild_date(value):
    value_str = str(value)  # Ensure it's a string
    if value_str.startswith('2024'):
        # Extract parts to form 'dd/mm/yy'
        return f"{value_str[-2:]}/{value_str[5:7]}/{value_str[2:4]}"
    return value_str  # Leave other values as is

def process_dates(value):
    value_str = str(value)  # Ensure the value is a string
    if value_str.startswith('2024'):
        return value_str[:10]  # Take the first 10 characters
    else:
        return value_str[:8]  # Take the first 8 characters

def map_berth(row):
    port = row['Port']
    berth = row['Berth']
    flag = ''  # Default flag value

    # Determine the mapping based on port
    if port == 'Bahia Blanca':
        mapped_berth = bahia_berth_mapping.get(berth, None)
    elif port == 'Rosario':
        mapped_berth = rosario_berth_mapping.get(berth, None)
    elif port == 'San Lorenzo':
        mapped_berth = san_lorenzo_berth_mapping.get(berth, None)
    elif port == 'Uruguay':
        mapped_berth = uruguay_berth_mapping.get(berth, None)
    elif port == 'Necochea':
        mapped_berth = 'Necochea'
    elif port == 'Nueva Palmira':
        mapped_berth = 'Nueva Palmira'
    elif port == 'Campana':
        mapped_berth = 'Carboclor'
    elif port == 'Diamante':
        mapped_berth = 'Diamante'
    elif port == 'Parana':
        mapped_berth = 'Parana'
    elif port == 'Ramallo':
        mapped_berth = 'Ramallo'
    elif port == 'Rio Grande':
        mapped_berth = 'Rio Grande'
    elif port == 'San Nicolas':
        mapped_berth = 'Elevador San Nicolas'
    elif port == 'San Pedro':
        mapped_berth = 'Elevador San Pedro'
    elif port == 'Zarate':
        mapped_berth = 'Del Guazu'
    elif port == 'Buenos Aires':
        mapped_berth = 'Terbasa 1'
    elif port == 'Alianza':
        mapped_berth = 'Alianza G2'
    elif port == 'Escobar':
        mapped_berth = 'Top off'
    elif port == 'Santa Fe':
        mapped_berth = 'Elevador'
    elif port == 'CONTAINERS':
        mapped_berth = 'Containers'
    elif port == 'Concep. Del Uruguay':
        mapped_berth = 'Concep. Del Uruguay'   
    else:
        mapped_berth = None  # Unhandled port

    # If no mapping is found, update the flag
    if mapped_berth is None:
        flag = 'To be checked'
        mapped_berth = berth  # Retain original berth

    return pd.Series([mapped_berth, flag])



## VEGOILS

### Vegoils Lineups NABSA

In [0]:
vegoils_lineups=sp_mgr.read_pd_from_excel('/sites/GRP-TradingLineups/Lineups/PROVIDERS/OILS/lineup.xlsx',sheet_name='DailyVesselLineUp',header=None)

# Locate the row where 'Port' is located
port_row_index = vegoils_lineups[vegoils_lineups.eq('Port').any(axis=1)].index[0]

vegoils_lineups.columns = vegoils_lineups.iloc[port_row_index]

vegoils_lineups = vegoils_lineups.iloc[port_row_index+1:].reset_index(drop=True)


oils = [
    "SOYBEAN OIL",
    "NEUTRAL OIL",
    "SUN FLOWER OIL",
    "LECITHIN",
    "SOYBEANOIL REFINED",
    "OLEIN",
    "GLYCERINE",
    "BIODIESEL",
    "VEGETABLE OIL",
    "TALLOW"
]

vegoils_lineups=vegoils_lineups[vegoils_lineups['Commodity'].isin(oils)]
vegoils_lineups.reset_index(drop=True)


# Apply function to DataFrame
vegoils_lineups['Date'] = vegoils_lineups.apply(extract_date, axis=1)
vegoils_lineups['Date'] = vegoils_lineups['Date'].fillna(method='bfill')


In [0]:
process=sp_mgr.read_pd_from_excel('/sites/GRP-TradingLineups/Lineups/ANNEX/NABSA.xlsx')
process['NABSA']=process['NABSA'].str.lower()
vegoils_lineups['Commodity']=vegoils_lineups['Commodity'].str.lower()
vegoils_lineups.rename(columns={'Origin':'Origin2'},inplace=True)

merged=pd.merge(vegoils_lineups,process,left_on="Commodity",right_on="NABSA",how="left")
merged.drop(columns=['Commodity','NABSA','ALPEMAR','Old LineUp','Origin','Description'],inplace=True)
merged.rename(columns={'Origin2':'Origin'},inplace=True)
vegoils_lineups=merged

vegoils_lineups['Month']=vegoils_lineups['Date'].dt.month
vegoils_lineups['Year']=vegoils_lineups['Date'].dt.year

vegoils_lineups['Region']=np.nan
vegoils_lineups['Coordinator']=np.nan
vegoils_lineups['Status']='ANNOUNCED'

vegoils_lineups['Coordinator']=vegoils_lineups['Charterer']

vegoils_lineups=vegoils_lineups[['Date','Month','Year','Status','Origin','Process','Product','Group','Vessel','Charterer','Coordinator','Port','Terminal','Destination','Region','Tons']]
vegoils_lineups=vegoils_lineups.rename(columns={'Charterer':'Shipper','Terminal':'Berth','Tons':'Quantity','Commodity':'Product'})


vegoils_lineups['Port']=vegoils_lineups['Port'].str.rstrip()
vegoils_lineups['Berth']=vegoils_lineups['Berth'].str.lower()
vegoils_lineups['Berth']=vegoils_lineups['Berth'].str.rstrip()

vegoils_lineups['Destination']=vegoils_lineups['Destination'].str.upper()
vegoils_lineups['Destination'] = vegoils_lineups['Destination'].replace(country_mapping)


vegoils_lineups['Port'] = vegoils_lineups['Port'].replace(port_mapping)

if 'Flag' not in vegoils_lineups.columns:
    vegoils_lineups['Flag'] = ''  
    
vegoils_lineups[['Berth', 'Flag']] = vegoils_lineups.apply(map_berth, axis=1)

vegoils_lineups['Origin']=vegoils_lineups['Origin'].replace(origins_mapping)
vegoils_lineups['Destination']=vegoils_lineups['Destination'].replace(country_mapping)
vegoils_lineups['Region']=vegoils_lineups['Destination'].replace(country_region_mapping)


### VEGOILS SAILED ALPEMAR

In [0]:
# Read the Excel file
new = sp_mgr.read_pd_from_excel('/sites/GRP-TradingLineups/Lineups/PROVIDERS/OILS/Shipments.xls')

# Identify the index of the 'Commodity Group' row
port_row_index = new[new.eq('Commodity Group').any(axis=1)].index[0]

# Set the new column headers and drop rows above
new.columns = new.iloc[port_row_index]
new = new.iloc[port_row_index + 1:].reset_index(drop=True)

# Define the pattern
pattern = r"Ref [A-Za-z]+ 202[45]:"

# Handle potential NaN values in the 'Commodity Group' column
new['Commodity Group'] = new['Commodity Group'].fillna('')

# Check if any row matches the pattern
matches = new['Commodity Group'].str.contains(pattern, regex=True)

if matches.any():
    # Find the index of the first matching row
    ref_index = new[matches].index[0]
    # Filter rows above this index
    new = new.iloc[:ref_index]
else:
    # If no match exists, retain the whole DataFrame
    new = new

# Filter for VegOil and Liquid Bulk
vegoils = new[(new['Commodity Group'] == 'VegOil') | (new['Commodity Group'] == 'Liquid Bulk')]


In [0]:

process=sp_mgr.read_pd_from_excel('/sites/GRP-TradingLineups/Lineups/ANNEX/vegoils process.xlsx')
process['ALPEMAR']=process['ALPEMAR'].str.lower()
vegoils['Commodity']=vegoils['Commodity'].str.lower()
merged=pd.merge(vegoils,process,left_on="Commodity",right_on="ALPEMAR",how="left")
merged.drop(columns=['Commodity','ALPEMAR'],inplace=True)

merged['Op.']='SAILED'

merged['Unmoored'] = merged['Unmoored'].str[:10]

# Convert to datetime format
merged['Unmoored'] = pd.to_datetime(merged['Unmoored'], format='%d/%m/%Y', errors='coerce')

merged['Month']=merged['Unmoored'].dt.month
merged['Year']=merged['Unmoored'].dt.year

merged = merged.rename(columns={"Unmoored": "Date",'Op.':'Status','Tons':'Quantity'})

shippers=sp_mgr.read_pd_from_excel('/sites/GRP-TradingLineups/Lineups/ANNEX/shippers.xlsx')
shippers = shippers.drop_duplicates(subset=['Shipper/coordinator'])

merged=pd.merge(merged,shippers,left_on="Shipper",right_on="Shipper/coordinator",how="left")
merged.drop(columns=['Shipper','Shipper/coordinator'],inplace=True)
merged = merged.rename(columns={"Shipper/coordinator2": "Shipper"})

merged=pd.merge(merged,shippers,left_on="Coord.",right_on="Shipper/coordinator",how="left")
merged.drop(columns=['Coord.','Shipper/coordinator'])
merged = merged.rename(columns={"Shipper/coordinator2": "Coordinator"})
merged['Region']=np.nan
merged_new=merged[['Date','Month','Year', 'Status','Origin','Process','Product','Group', 'Vessel','Shipper', 'Coordinator','Port', 'Berth','Destination','Region','Quantity']]
merged_new['Port']=merged_new['Port'].str.rstrip()
merged_new['Berth']=merged_new['Berth'].str.lower()
merged_new['Berth']=merged_new['Berth'].str.rstrip()

merged_new['Destination']=merged_new['Destination'].str.upper()
merged_new['Destination'] = merged_new['Destination'].replace(country_mapping)


merged_new['Port'] = merged_new['Port'].replace(port_mapping)

if 'Flag' not in merged_new.columns:
    merged_new['Flag'] = ''  
    
merged_new[['Berth', 'Flag']] = merged_new.apply(map_berth, axis=1)

merged_new['Origin']=merged_new['Origin'].replace(origins_mapping)
merged_new['Destination']=merged_new['Destination'].replace(country_mapping)
merged_new['Region']=merged_new['Destination'].replace(country_region_mapping)

vegoils_combined=pd.concat([merged_new,vegoils_lineups])
vegoils_combined['Category']='OILS'

sp_mgr.save_pd_to_excel('/sites/GRP-TradingLineups/Lineups/PROVIDERS\OILS\Vegoils lineups.xlsx',vegoils_combined,index=False)

## BEANS

### URUGUAY

#### Montevideo

In [0]:
montevideo=sp_mgr.read_pd_from_excel('/sites/GRP-TradingLineups/Lineups/PROVIDERS/BEANS/LINE UP MONTEVIDEO.xlsx')


port_row_index = montevideo[montevideo.eq('VSLS').any(axis=1)].index[0]

montevideo.columns = montevideo.iloc[port_row_index]

montevideo = montevideo.iloc[port_row_index+1:].reset_index(drop=True)


# Find the index of the row with "GRAND TOTAL" in the first column
split_index = montevideo[montevideo["VSLS"] == "GRAND TOTAL:"].index

if not split_index.empty:
    # Split the DataFrame into announced and sailed
    split_index = split_index[0]
    announced = montevideo.iloc[:split_index]
    sailed = montevideo.iloc[split_index + 1:]
else:
    # If "GRAND TOTAL" is not found, assign the entire DataFrame to announced and leave sailed empty
    announced = montevideo
    sailed = pd.DataFrame()

# Select only the first 16 columns
announced = announced.iloc[:, :16]
sailed = sailed.iloc[:, :16]


sailed=sailed[sailed['COMMODITY']=='SOYBEANS']
sailed['ETF']=pd.to_datetime(sailed['ETF'])

current_month = datetime.now().month
current_year = datetime.now().year

# Filter rows where the date is in the current month and year
sailed_montevideo= sailed[(sailed["ETF"].dt.month == current_month) & (sailed["ETF"].dt.year == current_year)]

announced=announced[announced['COMMODITY']=='SOYBEANS']
announced['ETF']=pd.to_datetime(announced['ETF'])

current_month = datetime.now().month
current_year = datetime.now().year

# Filter rows where the date is in the current month and year
announced_montevideo= announced[(announced["ETF"].dt.month == current_month) & (announced["ETF"].dt.year == current_year)]
announced_montevideo['OPERATION']='ANNOUNCED'



#### Nueva Palmira

In [0]:
nueva_palmira=sp_mgr.read_pd_from_excel('/sites/GRP-TradingLineups/Lineups/PROVIDERS/BEANS/LINE UP NUEVA PALMIRA.xlsx')

port_row_index = nueva_palmira[nueva_palmira.eq('VSLS').any(axis=1)].index[0]

nueva_palmira.columns = nueva_palmira.iloc[port_row_index]

nueva_palmira = nueva_palmira.iloc[port_row_index+1:].reset_index(drop=True)


# Find the index of the row with "GRAND TOTAL" in the first column
split_index = nueva_palmira[nueva_palmira["VSLS"] == "GRAND TOTAL:"].index

if not split_index.empty:
    # Split the DataFrame into announced and sailed
    split_index = split_index[0]
    announced = nueva_palmira.iloc[:split_index]
    sailed = nueva_palmira.iloc[split_index + 1:]
else:
    # If "GRAND TOTAL" is not found, assign the entire DataFrame to announced and leave sailed empty
    announced = nueva_palmira
    sailed = pd.DataFrame()


sailed = sailed[(sailed['COMMODITY'] == 'SBS UY') | 
                      (sailed['COMMODITY'] == 'SBS PY') | 
                      (sailed['COMMODITY'] == 'SBS NON GMO')]


sailed['ETF'] = sailed['ETF'].astype(str)

def process_dates(value):
    value_str = str(value)  # Ensure the value is a string
    if value_str.startswith('2024'):
        return value_str[:10]  # Take the first 10 characters
    else:
        return value_str[:8]  # Take the first 8 characters

sailed['ETF']=sailed['ETF'].apply(process_dates)



# Apply the function to the column
sailed['ETF'] = sailed['ETF'].apply(rebuild_date)
def is_valid_date_string(value):
    return isinstance(value, str) and len(value) >= 6 and '-' not in value[-3:]  # crude check to skip malformed ones

# Apply to both announced and sailed if needed
sailed['ETF'] = sailed['ETF'].apply(lambda x: x if is_valid_date_string(x) else pd.NA)
sailed = sailed.dropna(subset=['ETF'])

sailed['ETF'] = pd.to_datetime(sailed['ETF'], format='%d/%m/%y', errors='coerce')

current_month = datetime.now().month
current_year = datetime.now().year

# Filter rows where the date is in the current month and year
sailed_nueva_palmira= sailed[(sailed["ETF"].dt.month == current_month) & (sailed["ETF"].dt.year == current_year)]


sailed_nueva_palmira["VSLS"] = sailed_nueva_palmira["CONVOYS"].where(sailed_nueva_palmira["CONVOYS"].notna() & (sailed_nueva_palmira["CONVOYS"] != ""), sailed_nueva_palmira["VSLS"])

sailed_nueva_palmira_gped = sailed_nueva_palmira.groupby(['VSLS', 'COMMODITY','ETF'], as_index=False).agg(
    {
        'CARGO': 'sum',
        'VSLS': 'last',
        'CONVOYS': 'last',
        'SELLER ': 'last',
        'SHIPPER / RECEIVER': 'last',
        'CHARTERER': 'last',
        'ETF': 'last',
        'DESTINATION / ORIGIN': 'last',
        'OPERATION': 'last',
        'AGENTS': 'last',
        'COMMENTS': 'last'
    }
)
sailed_nueva_palmira_gped['OPERATION']='SAILED'

announced_nueva_palmira=announced

announced_nueva_palmira = announced_nueva_palmira[(announced_nueva_palmira['COMMODITY'] == 'SBS UY') | 
                      (announced_nueva_palmira['COMMODITY'] == 'SBS PY') | 
                      (announced_nueva_palmira['COMMODITY'] == 'SBS NON GMO')]
announced_nueva_palmira['ETF'] = announced_nueva_palmira['ETF'].astype(str)
announced_nueva_palmira['ETF']=announced_nueva_palmira['ETF'].apply(process_dates)
announced_nueva_palmira['ETF'] = announced_nueva_palmira['ETF'].apply(rebuild_date)

announced_nueva_palmira['ETF']=pd.to_datetime(announced_nueva_palmira['ETF'], errors='coerce')

announced_nueva_palmira["VSLS"] = announced_nueva_palmira["CONVOYS"].where(announced_nueva_palmira["CONVOYS"].notna() & (announced_nueva_palmira["CONVOYS"] != ""), announced_nueva_palmira["VSLS"])
announced_nueva_palmira['OPERATION']='ANNOUNCED'

announced_nueva_palmira=announced_nueva_palmira[['VSLS','CONVOYS','CARGO','COMMODITY','SELLER ', 'SHIPPER / RECEIVER','CHARTERER','ETF','DESTINATION / ORIGIN','OPERATION','AGENTS','COMMENTS']]

uru_beans=pd.concat([sailed_montevideo,sailed_nueva_palmira_gped,announced_montevideo,announced_nueva_palmira])
uru_beans.reset_index(inplace=True)

uru_beans=uru_beans.rename(columns={"VSLS": "Vessel",'COMMODITY':'Product','CARGO':'Quantity','SHIPPER / RECEIVER':'Shipper','CHARTERER':'Coordinator','ETF':'Date','DESTINATION / ORIGIN':'Destination','OPERATION':'Status','COMMENTS':'Berth'})

uru_beans['Month']=uru_beans['Date'].dt.month
uru_beans['Year']=uru_beans['Date'].dt.year

uru_beans["Origin"] = uru_beans["Product"].str[-2:].apply(lambda x: x if x in ['UY', 'PY'] else 'UY')

uru_beans = uru_beans.assign(Process=pd.NA, Group=pd.NA, Port='URUGUAY', Region=pd.NA)

uru_beans=uru_beans[['Date','Month','Year','Status','Origin','Process','Product','Group', 'Vessel','Shipper', 'Coordinator','Port', 'Berth','Destination','Region','Quantity']]

uru_beans["Group"] = uru_beans["Product"].apply(lambda x: "NON GMO" if "NON GMO" in x else "")
uru_beans['Product']=uru_beans['Product'].str[:3]

### ARG

In [0]:
sailed_arg=sp_mgr.read_pd_from_excel('/sites/GRP-TradingLineups/Lineups/PROVIDERS/BEANS/sailed.xlsx',sheet_name='Sailed Vessels')
lineup_arg=sp_mgr.read_pd_from_excel('/sites/GRP-TradingLineups/Lineups/PROVIDERS/BEANS/lineup.xlsx',sheet_name='DailyVesselLineUp')

In [0]:
# Locate the row where 'Port' is located
port_row_index = sailed_arg[sailed_arg.eq('Port').any(axis=1)].index[0]

sailed_arg.columns = sailed_arg.iloc[port_row_index]

sailed_arg = sailed_arg.iloc[port_row_index+1:].reset_index(drop=True)


sailed_arg=sailed_arg[sailed_arg['Origin']=='ARGENTINA']
sailed_arg=sailed_arg[sailed_arg['Cargo']=='SOYA BEAN']


port_row_index = lineup_arg[lineup_arg.eq('Port').any(axis=1)].index[0]

lineup_arg.columns = lineup_arg.iloc[port_row_index]

lineup_arg = lineup_arg.iloc[port_row_index+1:].reset_index(drop=True)

lineup_arg=lineup_arg[lineup_arg['Origin']=='ARGENTINA']
lineup_arg=lineup_arg[lineup_arg['Commodity']=='SOYA BEAN']

arg_beans=pd.concat([sailed_arg,lineup_arg])

In [0]:
if arg_beans.empty:
    arg_beans = pd.DataFrame(columns=[
        'Date', 'Month', 'Year', 'Status', 'Origin', 'Process', 'Product', 'Group',
        'Vessel', 'Charterer', 'Coordinator', 'Port', 'Terminal', 'Destination',
        'Region', 'Tons'
    ])
    
arg_beans['Date'] = pd.to_datetime(arg_beans['Date'], format='%d/%m/%y')
arg_beans['Month']=arg_beans['Date'].dt.month
arg_beans['Year']=arg_beans['Date'].dt.year

arg_beans['Process']=np.nan
arg_beans['Group']=np.nan
arg_beans['Region']=np.nan
arg_beans['Product']='SBS'


In [0]:
arg_beans['Date'] = pd.to_datetime(arg_beans['Date'], format='%d/%m/%y')
arg_beans['Month']=arg_beans['Date'].dt.month
arg_beans['Year']=arg_beans['Date'].dt.year

arg_beans['Process']=np.nan
arg_beans['Group']=np.nan
arg_beans['Region']=np.nan
arg_beans['Product']='SBS'
#arg_beans['Coordinator']==arg_beans['Charterer']

arg_beans=arg_beans[['Date','Month','Year','Status','Origin','Process','Product','Group','Vessel','Charterer','Coordinator','Port','Terminal','Destination','Region','Tons']]
arg_beans=arg_beans.rename(columns={'Charterer':'Shipper','Terminal':'Berth','Tons':'Quantity'})

beans_lineups=pd.concat([arg_beans,uru_beans])

if not beans_lineups.empty:
    # Clean and transform data
    beans_lineups['Port'] = beans_lineups['Port'].str.rstrip()
    beans_lineups['Berth'] = beans_lineups['Berth'].str.lower().str.rstrip()
    beans_lineups['Destination'] = beans_lineups['Destination'].str.upper().replace(country_mapping)
    beans_lineups['Port'] = beans_lineups['Port'].replace(port_mapping)

    # Ensure Flag column exists
    if 'Flag' not in beans_lineups.columns:
        beans_lineups['Flag'] = ''  

    # Apply map_berth function
    beans_lineups[['Berth', 'Flag']] = beans_lineups.apply(map_berth, axis=1)

    beans_lineups['Origin'] = beans_lineups['Origin'].replace(origins_mapping)
    beans_lineups['Destination'] = beans_lineups['Destination'].replace(country_mapping)
    beans_lineups['Region'] = beans_lineups['Destination'].replace(country_region_mapping)
    beans_lineups['Product'] = 'SOYBEANS'
    beans_lineups['Category'] = 'BEANS'

# Save the (possibly empty) DataFrame
sp_mgr.save_pd_to_excel('/sites/GRP-TradingLineups/Lineups/PROVIDERS/BEANS/Beans_lineups.xlsx', beans_lineups, index=False)


## MEAL

### Nueva Palmira

In [0]:
nueva_palmira=sp_mgr.read_pd_from_excel('/sites/GRP-TradingLineups/Lineups/PROVIDERS/MEAL/LINE UP NUEVA PALMIRA.xlsx')

# Locate the row where 'Port' is located
port_row_index = nueva_palmira[nueva_palmira.eq('VSLS').any(axis=1)].index[0]

nueva_palmira.columns = nueva_palmira.iloc[port_row_index]

nueva_palmira = nueva_palmira.iloc[port_row_index+1:].reset_index(drop=True)

# Find the index of the row with "GRAND TOTAL" in the first column
split_index = nueva_palmira[nueva_palmira["VSLS"] == "GRAND TOTAL:"].index

if not split_index.empty:
    # Split the DataFrame into announced and sailed
    split_index = split_index[0]
    announced = nueva_palmira.iloc[:split_index]
    sailed = nueva_palmira.iloc[split_index + 1:]
else:
    # If "GRAND TOTAL" is not found, assign the entire DataFrame to announced and leave sailed empty
    announced = nueva_palmira
    sailed = pd.DataFrame()


sailed = sailed[(sailed['COMMODITY'] == 'SBM BOL') | 
                      (sailed['COMMODITY'] == 'SBM PY') | 
                      (sailed['COMMODITY'] == 'SBM ARG')| 
                      (sailed['COMMODITY'] == 'SBM')]


sailed['ETF'] = sailed['ETF'].astype(str)

sailed['ETF']=sailed['ETF'].apply(process_dates)

# Apply the function to the column
sailed['ETF'] = sailed['ETF'].apply(rebuild_date)
sailed['ETF'] = pd.to_datetime(sailed['ETF'], format='%d/%m/%y')

current_month = datetime.now().month
current_year = datetime.now().year

# Filter rows where the date is in the current month and year
sailed_nueva_palmira= sailed[(sailed["ETF"].dt.month == current_month) & (sailed["ETF"].dt.year == current_year)]


sailed_nueva_palmira["VSLS"] = sailed_nueva_palmira["CONVOYS"].where(sailed_nueva_palmira["CONVOYS"].notna() & (sailed_nueva_palmira["CONVOYS"] != ""), sailed_nueva_palmira["VSLS"])

sailed_nueva_palmira_gped = sailed_nueva_palmira.groupby(['VSLS', 'COMMODITY','ETF'], as_index=False).agg(
    {
        'CARGO': 'sum',
        'VSLS': 'last',
        'CONVOYS': 'last',
        'SELLER ': 'last',
        'SHIPPER / RECEIVER': 'last',
        'CHARTERER': 'last',
        'ETF': 'last',
        'DESTINATION / ORIGIN': 'last',
        'OPERATION': 'last',
        'AGENTS': 'last',
        'COMMENTS': 'last'
    }
)
sailed_nueva_palmira['OPERATION']='SAILED'

announced_nueva_palmira=announced

announced_nueva_palmira = announced_nueva_palmira[(announced_nueva_palmira['COMMODITY'] == 'SBM BOL') | 
                      (announced_nueva_palmira['COMMODITY'] == 'SBM PY') | 
                      (announced_nueva_palmira['COMMODITY'] == 'SBM ARG')|
                      (announced_nueva_palmira['COMMODITY'] == 'SBM')]

announced_nueva_palmira['ETF'] = announced_nueva_palmira['ETF'].astype(str)
announced_nueva_palmira['ETF']=announced_nueva_palmira['ETF'].apply(process_dates)
announced_nueva_palmira['ETF'] = announced_nueva_palmira['ETF'].apply(rebuild_date)

announced_nueva_palmira['ETF']=pd.to_datetime(announced_nueva_palmira['ETF'], errors='coerce')

announced_nueva_palmira["VSLS"] = announced_nueva_palmira["CONVOYS"].where(announced_nueva_palmira["CONVOYS"].notna() & (announced_nueva_palmira["CONVOYS"] != ""), announced_nueva_palmira["VSLS"])
announced_nueva_palmira['OPERATION']='ANNOUNCED'

announced_nueva_palmira=announced_nueva_palmira[['VSLS','CONVOYS','CARGO','COMMODITY','SELLER ', 'SHIPPER / RECEIVER','CHARTERER','ETF','DESTINATION / ORIGIN','OPERATION','AGENTS','COMMENTS']]

meal_nueva=pd.concat([sailed_nueva_palmira,announced_nueva_palmira])

meal_nueva=meal_nueva.rename(columns={"VSLS": "Vessel",'COMMODITY':'Product','CARGO':'Quantity','SHIPPER / RECEIVER':'Shipper','CHARTERER':'Coordinator','ETF':'Date','DESTINATION / ORIGIN':'Destination','OPERATION':'Status','COMMENTS':'Berth'})

meal_nueva['Month']=meal_nueva['Date'].dt.month
meal_nueva['Year']=meal_nueva['Date'].dt.year

meal_nueva["Origin"] = meal_nueva["Product"].str[-2:].apply(lambda x: x if x in ['UY', 'PY'] else 'UY')

meal_nueva = meal_nueva.assign(Process=pd.NA, Group=pd.NA, Port='URUGUAY', Region=pd.NA)

meal_nueva=meal_nueva[['Date','Month','Year','Status','Origin','Process','Product','Group', 'Vessel','Shipper', 'Coordinator','Port', 'Berth','Destination','Region','Quantity']]

meal_nueva['Product']=meal_nueva['Product'].str[:3]


meal_nueva['Port']=meal_nueva['Port'].str.rstrip()
meal_nueva['Berth']=meal_nueva['Berth'].str.lower()
meal_nueva['Berth']=meal_nueva['Berth'].str.rstrip()

meal_nueva['Destination']=meal_nueva['Destination'].str.upper()
meal_nueva['Destination'] = meal_nueva['Destination'].replace(country_mapping)


meal_nueva['Port'] = meal_nueva['Port'].replace(port_mapping)

if not meal_nueva.empty:
    if 'Flag' not in meal_nueva.columns:
        meal_nueva['Flag'] = ''  
        
    meal_nueva[['Berth', 'Flag']] = meal_nueva.apply(map_berth, axis=1)

    meal_nueva['Origin']=meal_nueva['Origin'].replace(origins_mapping)
    meal_nueva['Destination']=meal_nueva['Destination'].replace(country_mapping)
    meal_nueva['Region']=meal_nueva['Destination'].replace(country_region_mapping)

    meal_nueva['Category']='MEAL'


### Lineups

In [0]:
meal_announced=sp_mgr.read_pd_from_excel('/sites/GRP-TradingLineups/Lineups/PROVIDERS/MEAL/ANNCD-C.xls')


In [0]:
port_row_index = meal_announced[meal_announced.eq('Berth').any(axis=1)].index[0]

meal_announced.columns = meal_announced.iloc[port_row_index]

meal_announced = meal_announced.iloc[port_row_index+1:].reset_index(drop=True)


# Regular expression to detect flag values (e.g., 'PortName - Country')
flag_pattern = r'^[A-Za-z\s]+ - [A-Za-z\s]+$'

# Variable to hold the current port name
current_port = 'San Lorenzo'

for index, row in meal_announced.iterrows():
    vessel_value = row['Vessel']
    
    # Check if the value is a valid string and matches the flag pattern
    if isinstance(vessel_value, str) and re.match(flag_pattern, vessel_value):
        current_port = vessel_value.split(' - ')[0]  # Extract the port name before ' - '
    
    meal_announced.at[index, 'Port'] = current_port

meal_announced=meal_announced[(meal_announced['Commodity']=='Sbm')|(meal_announced['Commodity']=='Shp')|(meal_announced['Commodity']=='Sfmp')|(meal_announced['Commodity']=='Wpp')]

meal_announced['Berth']=meal_announced['Berth'].str.lower()
meal_announced['Berth']=meal_announced['Berth'].str.rstrip()



if 'Flag' not in meal_announced.columns:
    meal_announced['Flag'] = ''  

meal_announced[['Berth', 'Flag']] = meal_announced.apply(map_berth, axis=1)

meal_announced[['Date', 'Flag']] = meal_announced.apply(
    lambda row: pd.Series(extract_date_or_default_sbm(row, default_days=5)),
    axis=1
)


meal_announced.drop(columns=['Status'],inplace=True)
meal_announced['Date'] = pd.to_datetime(meal_announced['Date'], format='%d/%m/%Y', errors='coerce')
meal_announced['Month']=meal_announced['Date'].dt.month
meal_announced['Year']=meal_announced['Date'].dt.year
meal_announced['Status']='ANNOUNCED'

meal_announced=meal_announced.rename(columns={'Tons':'Quantity','Commodity':'Product','Destination/Origin':'Destination','Coord':'Coordinator'})
meal_announced['Shipper']=np.nan
meal_announced['Process']=np.nan
meal_announced['Group']=np.nan
meal_announced['Region']=np.nan

meal_announced['Origin']='ARG'
meal_announced=meal_announced[['Date','Month','Year','Status','Origin','Process','Product','Group','Vessel','Shipper','Coordinator','Port','Berth','Destination','Region','Quantity','Flag']]

meal_announced['Port']=meal_announced['Port'].str.rstrip()

meal_announced['Destination']=meal_announced['Destination'].str.upper()
meal_announced['Destination'] = meal_announced['Destination'].replace(country_mapping)


meal_announced['Port'] = meal_announced['Port'].replace(port_mapping)



    


meal_announced['Origin']=meal_announced['Origin'].replace(origins_mapping)
meal_announced['Destination']=meal_announced['Destination'].replace(country_mapping)
meal_announced['Region']=meal_announced['Destination'].replace(country_region_mapping)





### MEAL SAILED

In [0]:
# Read the Excel file
new = sp_mgr.read_pd_from_excel('/sites/GRP-TradingLineups/Lineups/PROVIDERS/MEAL/Shipments.xls')

# Identify the row containing 'Commodity Group'
port_row_index = new[new.eq('Commodity Group').any(axis=1)].index[0]

# Set headers and drop rows above
new.columns = new.iloc[port_row_index]
new = new.iloc[port_row_index + 1:].reset_index(drop=True)

# Define the pattern and handle NaN
new['Commodity Group'] = new['Commodity Group'].fillna('')  # Replace NaN with empty string
pattern = r"Ref [A-Za-z]+ 202[45]:"
matches = new['Commodity Group'].str.contains(pattern, regex=True)

if matches.any():
    ref_index = new[matches].index[0]
    new = new.iloc[:ref_index]

# Filter meals, ensuring no NaN issues
meals = ['Sbm', 'Sbm Par', 'Sbm 46,5', 'Shp', 'Sbm hp', 'Sbm super hp', 'Sfmp', 'Sbm 45,5', 'Sbm 44', 'Sbm Bol', 'Wpp','Sbm 42,5']
new['Commodity'] = new['Commodity'].fillna('')  # Replace NaN with empty string
meal_sailed = new[new['Commodity'].isin(meals)]

# Clean and transform meal_sailed
meal_sailed['Op.'] = 'SAILED'
meal_sailed['Unmoored'] = meal_sailed['Unmoored'].str[:10]
meal_sailed['Unmoored'] = pd.to_datetime(meal_sailed['Unmoored'], format='%d/%m/%Y', errors='coerce')
meal_sailed['Month'] = meal_sailed['Unmoored'].dt.month
meal_sailed['Year'] = meal_sailed['Unmoored'].dt.year

meal_sailed = meal_sailed.rename(columns={'Commodity': 'Product', "Unmoored": "Date", 'Op.': 'Status', 'Tons': 'Quantity'})

# Handle Shippers
shippers = sp_mgr.read_pd_from_excel('/sites/GRP-TradingLineups/Lineups/Annex/Shippers.xlsx')
shippers = shippers.drop_duplicates(subset=['Shipper/coordinator'])

# Merge shippers for "Shipper"
meal_sailed = pd.merge(meal_sailed, shippers, left_on="Shipper", right_on="Shipper/coordinator", how="left", suffixes=('', '_new'))
meal_sailed['Shipper'] = meal_sailed['Shipper/coordinator2'].fillna(meal_sailed['Shipper'])
meal_sailed.drop(columns=['Shipper/coordinator', 'Shipper/coordinator2'], inplace=True)

# Merge shippers for "Coordinator"
meal_sailed = pd.merge(meal_sailed, shippers, left_on="Coord.", right_on="Shipper/coordinator", how="left", suffixes=('', '_new'))
meal_sailed['Coordinator'] = meal_sailed['Shipper/coordinator2'].fillna(meal_sailed['Coord.'])
meal_sailed.drop(columns=['Coord.', 'Shipper/coordinator', 'Shipper/coordinator2'], inplace=True)

# Assign additional columns
meal_sailed['Process'] = meal_sailed['Product'].apply(lambda x: x if x.lower().startswith('sbm ') and len(x) > 4 else '')
meal_sailed['Group'] = np.nan
meal_sailed['Region'] = np.nan

meal_sailed['Origin'] = 'ARG'

meal_sailed.loc[meal_sailed['Shipper'].str.contains('ADM', case=False, na=False), 'Origin'] = 'PYG'

# Standardize data
meal_sailed = meal_sailed[['Date', 'Month', 'Year', 'Status', 'Origin', 'Process', 'Product', 'Group', 'Vessel', 'Shipper', 'Coordinator', 'Port', 'Berth', 'Destination', 'Region', 'Quantity']]
meal_sailed['Port'] = meal_sailed['Port'].str.rstrip()
meal_sailed['Berth'] = meal_sailed['Berth'].str.lower().str.rstrip()
meal_sailed['Destination'] = meal_sailed['Destination'].str.upper().replace(country_mapping)
meal_sailed['Port'] = meal_sailed['Port'].replace(port_mapping)

if 'Flag' not in meal_sailed.columns:
    meal_sailed['Flag'] = ''

meal_sailed[['Berth', 'Flag']] = meal_sailed.apply(map_berth, axis=1)
meal_sailed['Origin'] = meal_sailed['Origin'].replace(origins_mapping)
meal_sailed['Destination'] = meal_sailed['Destination'].replace(country_mapping)
meal_sailed['Region'] = meal_sailed['Destination'].replace(country_region_mapping)

# Combine with other meal data
meal_lineups = pd.concat([meal_sailed, meal_announced, meal_nueva], ignore_index=True)

# Adjust Process and Origin for specific Products
meal_lineups.loc[meal_lineups['Process'] == 'Sbm Par', ['Process', 'Origin']] = ['', 'PYG']
meal_lineups.loc[meal_lineups['Process'] == 'Sbm Bol', ['Process', 'Origin']] = ['', 'BOL']

# Standardize Product names
sbm_replace = ['Sbm 46,5', 'Sbm Par', 'Sbm Bol', 'Sbm 45,5', 'Sbm 44', 'Sbm hp', 'Sbm super hp', 'Sbm','Sbm 42,5']
meal_lineups['Product'] = meal_lineups['Product'].apply(lambda x: 'SBM' if x in sbm_replace else x)
meal_lineups.loc[meal_lineups['Process'] == '', 'Process'] = 'SBM 46,5'
meal_lineups.loc[meal_lineups['Product'] == 'Shp', 'Product'] = 'SBPHULLS'
meal_lineups.loc[meal_lineups['Product'] == 'Sfmp', 'Product'] = 'SUNPEL'
meal_lineups.loc[meal_lineups['Product'] == 'Wpp', 'Product'] = 'WPP'

meal_lineups['Process'] = meal_lineups['Process'].apply(
    lambda x: x.replace('Sbm', 'SBM') if isinstance(x, str) and 'Sbm' in x else x
)
meal_lineups['Category'] = 'MEAL'
meal_lineups.loc[meal_lineups['Shipper'] == 'ADM', 'Origin'] = 'PGY'
# Save to Excel
sp_mgr.save_pd_to_excel('/sites/GRP-TradingLineups/Lineups/PROVIDERS/MEAL/meal_lineups.xlsx', meal_lineups, index=False)


## GRAINS

In [0]:
grains=sp_mgr.read_pd_from_excel('/sites/GRP-TradingLineups/Lineups/PROVIDERS/GRAINS/GRAIN-SBS-BARLEY-MALT SHIPMENTS.xls')

port_row_index = grains[grains.eq('CARGO').any(axis=1)].index[0]

grains.columns = grains.iloc[port_row_index]

grains = grains.iloc[port_row_index+1:].reset_index(drop=True)

ref_index = None

if 'AT ROADS' in grains['CARGO'].values:
    # Find the index of the row containing "AT ROADS"
    ref_index = grains[grains['CARGO'] == "AT ROADS"].index[0]
    # Filter rows above this index
    grains_sailed = grains.iloc[:ref_index-1]
else:
    # If the value doesn't exist, retain the whole DataFrame
    grains_sailed = grains

grains_sailed=grains_sailed.iloc[2:]

grains_sailed=grains_sailed.rename(columns={	'CARGO':'Product','VESSEL':'Vessel','PORT':'Port','BERTH':'Berth','ETA/ETF':'Date','TONS':'Quantity','SHIPPER':'Shipper','COORD':'Coordinator','DEST':'Destination'})

grains_sailed['Date']=grains_sailed['Date'].str[:10]
grains_sailed['Date'] = pd.to_datetime(grains_sailed['Date'], format='%d/%m/%Y', errors='coerce')
grains_sailed['Month']=grains_sailed['Date'].dt.month
grains_sailed['Year']=grains_sailed['Date'].dt.year
grains_sailed['Status']='SAILED'
grains_sailed['Origin']='ARG'
grains_sailed['Process']=np.nan
grains_sailed['Group']=np.nan
grains_sailed['Region']=np.nan

grains_sailed[['Process', 'Product']] = grains_sailed['Product'].str.extract(r'(.*?)\s*(.*)', expand=True).fillna("")

grains_sailed=grains_sailed[['Date','Month','Year','Status','Origin','Process','Product','Group','Vessel','Shipper','Coordinator','Port','Berth','Destination','Region','Quantity']]


grains_sailed['Port']=grains_sailed['Port'].str.rstrip()
grains_sailed['Berth']=grains_sailed['Berth'].str.lower()
grains_sailed['Berth']=grains_sailed['Berth'].str.rstrip()

grains_sailed['Destination']=grains_sailed['Destination'].str.upper()
grains_sailed['Destination'] = grains_sailed['Destination'].replace(country_mapping)


grains_sailed['Port'] = grains_sailed['Port'].replace(port_mapping)

if 'Flag' not in grains_sailed.columns:
    grains_sailed['Flag'] = ''  
    
grains_sailed[['Berth', 'Flag']] = grains_sailed.apply(map_berth, axis=1)

grains_sailed['Origin']=grains_sailed['Origin'].replace(origins_mapping)
grains_sailed['Destination']=grains_sailed['Destination'].replace(country_mapping)
grains_sailed['Region']=grains_sailed['Destination'].replace(country_region_mapping)


grains_announced=grains[ref_index:]
grains_announced = grains_announced .dropna(thresh=grains_announced .shape[1] - 3)


# Apply the function to the column
grains_announced['Date'] = grains_announced['ETA/ETF'].apply(lambda x: extract_date_or_default(x, default_days=5))
grains_announced['Date'] = pd.to_datetime(grains_announced['Date'], format='%d/%m/%Y', errors='coerce')

grains_announced['Month']=grains_announced['Date'].dt.month
grains_announced['Year']=grains_announced['Date'].dt.year


grains_announced=grains_announced.rename(columns={'CARGO':'Product','VESSEL':'Vessel','PORT':'Port','BERTH':'Berth','TONS':'Quantity','SHIPPER':'Shipper','COORD':'Coordinator','DEST':'Destination'})

grains_announced['Status']='ANNOUNCED'
grains_announced['Origin']='ARG'
grains_announced['Process']=np.nan
grains_announced['Group']=np.nan
grains_announced['Region']=np.nan

grains_announced=grains_announced[['Date','Month','Year','Status','Origin','Process','Product','Group','Vessel','Shipper','Coordinator','Port','Berth','Destination','Region','Quantity']]


grains_announced['Port']=grains_announced['Port'].str.rstrip()
grains_announced['Berth']=grains_announced['Berth'].str.lower()
grains_announced['Berth']=grains_announced['Berth'].str.rstrip()

grains_announced['Destination']=grains_announced['Destination'].str.upper()
grains_announced['Destination'] = grains_announced['Destination'].replace(country_mapping)


grains_announced['Port'] = grains_announced['Port'].replace(port_mapping)

if 'Flag' not in grains_announced.columns:
    grains_announced['Flag'] = ''  
    
grains_announced[['Berth', 'Flag']] = grains_announced.apply(map_berth, axis=1)

grains_announced['Origin']=grains_announced['Origin'].replace(origins_mapping)
grains_announced['Destination']=grains_announced['Destination'].replace(country_mapping)
grains_announced['Region']=grains_announced['Destination'].replace(country_region_mapping)

grains_lineups=pd.concat([grains_sailed,grains_announced])
grains_lineups['Category']='GRAINS'

grains_lineups.loc[(grains_lineups['Region'] == 'SOUTH AMERICA') & (grains_lineups['Product'] == 'Barley'), ['Process']] = 'MALTING'
grains_lineups.loc[(grains_lineups['Product'] == 'Maize'), ['Product']] = 'Corn'
grains_lineups = grains_lineups[grains_lineups['Product'] != 'Sbs']

sp_mgr.rm('/sites/GRP-TradingLineups/Lineups/PROVIDERS/GRAINS\Grains lineups.xlsx')
sp_mgr.save_pd_to_excel('/sites/GRP-TradingLineups/Lineups/PROVIDERS/GRAINS\Grains lineups.xlsx',grains_lineups,index=False)


In [0]:
grains.CARGO.unique()

In [0]:
if 'AT ROADS' in grains['ETA/ETF'].values:
    # Find the index of the row containing "AT ROADS"
    ref_index = grains[grains['ETA/ETF'] == "AT ROADS"].index[0]
    # Filter rows above this index
    grains_sailed = grains.iloc[:ref_index-1]

## MERGER LINEUPS

In [0]:
lineups_merged=pd.concat([vegoils_combined,beans_lineups,meal_lineups,grains_lineups])

lineups_merged['Month_Abbrev'] = lineups_merged['Month'].map(month_mapping)

lineups_merged['Shipper'] = lineups_merged['Shipper'].str.upper()
lineups_merged['Coordinator'] = lineups_merged['Coordinator'].str.upper()

lineups_merged['Shipper'] = lineups_merged['Shipper'].map(lambda x: shippers_mapping.get(x, x))
lineups_merged['Coordinator'] = lineups_merged['Coordinator'].map(lambda x: shippers_mapping.get(x, x))

lineups_merged.loc[(lineups_merged['Berth'].isna()) & (lineups_merged['Status'] == 'ANNOUNCED'), 'Flag'] = ''

lineups_merged['Date'] = pd.to_datetime(lineups_merged['Date'])

# Sort the DataFrame by Vessel and Date
lineups_merged = lineups_merged.sort_values(by=['Vessel', 'Date']).reset_index(drop=True)

# Initialize a column to store the adjusted dates
lineups_merged['Adjusted_Date'] = lineups_merged['Date']

# Define a time threshold (e.g., 7 days)
time_threshold = timedelta(days=7)

# Iterate through vessels to adjust close dates
# Iterate through vessels to adjust close dates only if the Port is the same
for vessel in lineups_merged['Vessel'].unique():
    vessel_data = lineups_merged[lineups_merged['Vessel'] == vessel]
    for i in range(1, len(vessel_data)):
        current_idx = vessel_data.index[i]
        prev_idx = vessel_data.index[i - 1]
        
        # Check if the Port is the same and the date is within the threshold
        if (vessel_data.loc[current_idx, 'Port'] == vessel_data.loc[prev_idx, 'Port'] and
            vessel_data.loc[current_idx, 'Date'] - vessel_data.loc[prev_idx, 'Adjusted_Date'] <= time_threshold):
            
            # Update the previous adjusted date to the maximum of the two
            max_date = max(vessel_data.loc[current_idx, 'Date'], vessel_data.loc[prev_idx, 'Adjusted_Date'])
            lineups_merged.loc[prev_idx, 'Adjusted_Date'] = max_date


# Update the main Date column with Adjusted_Date
lineups_merged['Date'] = lineups_merged['Adjusted_Date']
lineups_merged = lineups_merged.drop(columns=['Adjusted_Date'])  # Remove helper column if not needed 
lineups_merged['Month'] = lineups_merged['Date'].dt.month
lineups_merged['Quantity'] = pd.to_numeric(lineups_merged['Quantity'], errors='coerce')

sp_mgr.rm('/sites/GRP-TradingLineups/Lineups/merged/Lineups_merged.xlsx')
sp_mgr.save_pd_to_excel('/sites/GRP-TradingLineups/Lineups/merged/Lineups_merged.xlsx', lineups_merged, index=False)


In [0]:
# body_1 = """
# <p>The lineups have been generated. Please review them and make any necessary corrections directly in the Excel file.</p>
# Link to the Sharepoint:
# https://ldcom365.sharepoint.com/sites/GRP-TradingLineups/Lineups/Forms/AllItems.aspx

# The Lineup file to review is saved in the folder named Merged: Lineups_merged.xlsx

# <p><strong>IN CASE YOU HAVE TO MAKE ANY CHANGES (e.g., in a column, adding a row, etc.):</strong></p>

# <p>Please place an "x" in the "Flag" column. The row will then be recognized as manually modified and will not be erased the next time the code runs.</p>

# <p>If you encounter duplicate or nonsensical lines, you may simply delete them.</p>
# """

# adrss=["florian.girardi-ext@ldc.com","lucas.fazio@ldc.com","FERNANDO.CORREAURQUIZA@ldc.com","Lionel.Occhione@LDC.com","Mateo.Vergniaud@ldc.com","juan.carnemolla@LDC.com","valentin.chiesa@ldc.com","michelle.lambrechts@ldc.com","SANTIAGO.CARBAJAL@ldc.com","franco.gambetta@ldc.com","lucas.bel@ldc.com"]

# adrss_test=["florian.girardi-ext@ldc.com"]

# #LDCDataAccessLayerPy.mail.mail_send(to=adrss, subject=f'Lineups {datetime.now().strftime("%y-%m")}',from_addr="florian.girardi-ext@ldc.com",body=body_1,mime_type="html")
# # Send email with embedded chart and table
# LDCDataAccessLayerPy.mail.mail_send(
#     to=adrss_test,
#     subject=f'Lineups {datetime.now().strftime("%y-%m")}',
#     from_addr="florian.girardi-ext@ldc.com",
#     body=body_1,
#     mime_type="html",
# )
